##AUTOENCODER SCRIPT = this will perform a dimensionality reduction on both GH administed control data and all athlete data
ensure to activate uv environment in powershell: .\.venv\Scripts\activate.ps1

In [ ]:
#Autoencoder works on all data GHadmin. data and the athlete data itself = merged_df 

import tensorflow as tf
from tensorflow.keras import layers, models
import pandas as pd
from config import data as dcfg
import pandas as pd
from config import model as mcfg
import matplotlib.pyplot as plt

#import matplotlib.pyplot as plt


Import Data ising panda 
Import Config class to reference the path to the cleaned GH data
The autoencoder can not process non-numerical values - therefore they need to be removed and stored in another variable: ids (identies)

In [21]:
def proc(merged_df): # Use the path from your config file
    df = pd.read_csv(dcfg.merged_df)
    
    # 1. Isolate IDs (Column 0) 
    ids = df.iloc[:, 0]
    
    # 2. Drop the "id" column to leave  4 metrics
    numeric_df = df.drop(columns=["id"])
    
    # 3. Convert to Tensor then to NumPy float32
    df_tensor = tf.convert_to_tensor(numeric_df, dtype=tf.float32).numpy()
    
    # 4. Get the number of features = (4)
    n_features = df_tensor.shape[1]
    
    print(f"Data ready. Rows: {df_tensor.shape[0]}, Features: {n_features}")
    
    return df_tensor, n_features, ids


now that the dataset has been read in, and cleaned from previous R work, The autoencoder needs an input:
This is going to be a tabular outlier autoencoder 

Building and Training the Autoencoder - the input must always match the output - it is essentially recreating the dataset with its own found knowledge of what is normal vs what isn't

1. Input: 4 (Sex, Age, PNP, IGF)

2. Bottleneck: 3 (Your "Visualization" coordinates)

3. Output: 4 (Reconstructed Sex, Age, PNP, IGF)

Our bottleneck vector will be  TUPLE - this means that it is enclosed in ROUND brackets - it can not be changed once made
    - Different from a list which is enclosed in SQUARE brackets and is able to be modified.


In [ ]:
## BUILDING
def autoenc_build(input_shape, dim):
    model = models.Sequential([
        # Encoder part
        layers.Input(shape=(input_shape,)), 
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(dim, name='bottleneck'), # Your 3D coordinates
        
        # Decoder part (Note: these must be inside the [ ] brackets)
        layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(input_shape, activation="linear") # Back to 4 features
    ])
    return model

## COMPILING 
def compiler(autoencoder):
    autoencoder.compile(
        optimizer="Adam",
        loss='mse',
        metrics=['mae'] 
    )

## TRAINING
def train_autoencoder(autoencoder, epochs=None):
    # If you didn't pass a specific number, use the config file
    if epochs is None:
        epochs = mcfg.epoch 

    return autoencoder.fit(
        x=df_tensor, 
        y=df_tensor, 
        epochs=epochs, # Use the 'epochs' variable we just set
        validation_split=0.2
    )

Obtaining layer outputs (Middle of the endoer layer outputs) - this is the compressed co-ordiantes for each athlete
autoencoder - 4 in - 3 hidden - 4 out 
The encoder is the top half of the model - this is the 4 in and 3 out 
     - inside the 'latent space vectors' is a new matrix (numpy array)
     - it will have a shape of (99,3)
     - [rows, columns ] - this is the format of co=ordinates in python 


In [ ]:
def elbow_plot( input_shape, dim):
    dims = range(1, dim+1)
    errors = []
    for i in dims:
        model = autenc_build(input_shape, latent_dim = i),
        compiler(model)

        history = train_autoencoder(model, epochs = mcgf.epochs_2)
        final_loss_value = history.history['loss'][-1]
        errors.append(final_loss_value)
        return(dims, errors)

def plotting (dims, errors):
# dims - the dimensions you use, and final loss values - is from the loss function
    dims, final_loss_values = elbow_plot(n_features, mcfg.dim)

    plt.plot(dims, errors, colour = 'b', linestyle = '-', marker ='o')
    plt.title("Autoencoder elbow plot: finding optimal latent dimensions")
    plt.xlabel("Number of dimensions")
    plt.ylabel("Reconstruction Error (MSE)")
    plt.grid(True)
    plt.show()
    


In [ ]:

latent_space_vectors = bottleneck.predict(df_tensor)
#Reattach the volunteer ID df to the df processed by the autoencoder in order to ensure that all are back to being labelled - do not shuffle the data.

print(f"obtained {latent_vectors.shape[0]} latent vectors of dimensions {latent_vectors.shape[1]}")